## PyINE datasets visualization/demo

This notebook parses and displays samples/stats for a dataset of execution traces and trace deltas generated by our proposed framework.

In [ ]:
import collections

import matplotlib.pyplot as plt
import numpy as np

import pyine.data.deltas.dataset_reader
import pyine.data.deltas.dataset_utils
import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.utils.code.blocks
import pyine.utils.code.execution
import pyine.utils.portability

In [ ]:
target_source_dataset = "TACO"  # name of the dataset whose traces+deltas we should visualize below

In [ ]:
trace_reader = pyine.data.traces.dataset_reader.DatasetReader(
    lmdb_path=pyine.data.traces.dataset_utils.get_latest_dataset_path(target_source_dataset),
)
print(f"{target_source_dataset} traces dataset contains {len(trace_reader)} traces")
# the lmdb dataset stores a ton of metadata related to when/how it was created; let's display some of it
metadata = trace_reader.get_metadata()
metadata_fields_too_big_to_print = ["key_map", "installed_packages"]
print("traces dataset metadata:")
for metadata_key, metadata_value in metadata.items():
    if metadata_key in metadata_fields_too_big_to_print:  # too big to print, skip it
        continue
    print(f"\t{metadata_key}: {metadata_value}")

In [ ]:
# below, we will iterate over all traced solutions, gather some stats, and print them
traced_solution_tags = collections.Counter()
traced_function_count = 0
traced_block_count = 0
traced_line_count = 0
trace_desc_lengths = []  # in characters, approx, reference for comparisons w/ deltas

# for each traced solution in the dataset
for trace_idx, trace_data in enumerate(trace_reader):
    # get the parent problem data for this specific solution
    problem_data = trace_reader.get_problem_data(trace_idx)
    # extract traced steps that are 'in-context', i.e. inside the solution code string
    traced_steps = [t for t in trace_data.traced_steps if t is not None]
    traced_source_lines: set[int] = set()
    for t in traced_steps:
        if t.trace_key.file == pyine.utils.code.execution.EXEC_TRACE_FILE_NAME:
            traced_source_lines.add(t.trace_key.line)
        curr_desc_length = (  # sum up the length (in chars) required to repr all vars/args for the step
            len(t.global_variables.__repr__())
            + len(t.local_variables.__repr__())
            + len(t.arguments.__repr__())
            + len(t.return_value.__repr__())
            + len(t.exception.__repr__())
        )
        trace_desc_lengths.append(curr_desc_length)
    traced_line_count += len(traced_source_lines)
    for code_block_key_repr, code_block_data in trace_data.code_blocks.items():
        code_block_key = pyine.utils.code.execution.TraceKey.from_string(code_block_key_repr)
        assert code_block_key.file == pyine.utils.code.execution.EXEC_TRACE_FILE_NAME
        if code_block_key.line not in traced_source_lines:
            continue
        traced_block_count += 1
        if code_block_data.type == pyine.utils.code.blocks.BlockType.FUNCTION:
            traced_function_count += 1
    # gather and count tags for the current traced solution
    traced_solution_tags.update(problem_data.problem_tags)


print(f"{traced_function_count=}")
print(f"{traced_block_count=}")
print(f"{traced_line_count=}")

In [ ]:
plt.figure(figsize=(12, 12))

# plot distribution of tags
plt.subplot(2, 1, 1)
tags = list(traced_solution_tags.keys())
counts = list(traced_solution_tags.values())
plt.bar(tags, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Tags")
plt.ylabel("Frequency (log-scaled)")
plt.yscale("log")
plt.title("Distribution of found tags")

# plot distribution of trace step description lengths (in char counts)
plt.subplot(2, 1, 2)
plt.hist(trace_desc_lengths, bins=50)
plt.xlabel("Description length (chars)")
plt.ylabel("Frequency (log-scaled)")
plt.yscale("log")
plt.title("Distribution of trace step description lengths")
stats_text = f"Mean: {np.mean(trace_desc_lengths):.1f}\n"
stats_text += f"Min: {np.min(trace_desc_lengths)}\n"
stats_text += f"Max: {np.max(trace_desc_lengths)}\n"
stats_text += f"Std: {np.std(trace_desc_lengths):.1f}"
plt.text(
    0.95,
    0.95,
    stats_text,
    transform=plt.gca().transAxes,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(facecolor="white", alpha=0.8),
)
plt.tight_layout()
plt.show()

In [ ]:
# plot distributions for tags with prefixes: subset:, difficulty:, source:
subset_tags = {k: v for k, v in traced_solution_tags.items() if k.startswith("subset:")}
difficulty_tags = {k: v for k, v in traced_solution_tags.items() if k.startswith("difficulty:")}
source_tags = {k: v for k, v in traced_solution_tags.items() if k.startswith("source:")}

plt.figure(figsize=(18, 5))

# subset:*
plt.subplot(1, 3, 1)
tags = list(subset_tags.keys())
counts = list(subset_tags.values())
plt.bar(tags, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("subset:*")
plt.ylabel("Count")
plt.title("Distribution of subset tags")

# difficulty:*
plt.subplot(1, 3, 2)
tags = list(difficulty_tags.keys())
counts = list(difficulty_tags.values())
plt.bar(tags, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("difficulty:*")
plt.ylabel("Count")
plt.title("Distribution of difficulty tags")

# source:*
plt.subplot(1, 3, 3)
tags = list(source_tags.keys())
counts = list(source_tags.values())
plt.bar(tags, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("source:*")
plt.ylabel("Count")
plt.title("Distribution of source tags")

plt.tight_layout()
plt.show()

In [ ]:
# now, display some stats about the deltas for this dataset
deltas_reader = pyine.data.deltas.dataset_reader.DatasetReader(
    lmdb_path=pyine.data.deltas.dataset_utils.get_latest_dataset_path(target_source_dataset),
)
print(f"deltas dataset contains {len(deltas_reader)} traces")
delta_generator = deltas_reader.get_metadata()["delta_generator"]
print(f"deltas dataset was generated using '{delta_generator}' generator")

In [ ]:
# below, we will iterate over all deltas, gather some stats, and print them
delta_type_counter = collections.Counter()
delta_desc_lengths = []

# for each traced solution in the dataset
for trace_idx, trace_deltas in enumerate(deltas_reader):
    assert isinstance(trace_deltas, pyine.data.deltas.dataset_utils.TraceDeltaList)
    for delta in trace_deltas:
        assert isinstance(delta, pyine.data.deltas.dataset_utils.TraceDelta)
        delta_type_counter[delta.event_relationship] += 1
        delta_desc_lengths.append(len(delta.variables.__repr__()))

plt.figure(figsize=(12, 12))

# plot distribution of delta types
plt.subplot(2, 1, 1)
types = list(delta_type_counter.keys())
counts = list(delta_type_counter.values())
plt.bar(types, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Types")
plt.ylabel("Frequency (log-scaled)")
plt.yscale("log")
plt.title("Distribution of delta types")

# plot distribution of delta description lengths (in char counts)
plt.subplot(2, 1, 2)
plt.hist(delta_desc_lengths, bins=50)
plt.xlabel("Description length (chars)")
plt.ylabel("Frequency (log-scaled)")
plt.yscale("log")
plt.title("Distribution of delta description lengths")
stats_text = f"Mean: {np.mean(delta_desc_lengths):.1f}\n"
stats_text += f"Min: {np.min(delta_desc_lengths)}\n"
stats_text += f"Max: {np.max(delta_desc_lengths)}\n"
stats_text += f"Std: {np.std(delta_desc_lengths):.1f}"
plt.text(
    0.95,
    0.95,
    stats_text,
    transform=plt.gca().transAxes,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(facecolor="white", alpha=0.8),
)
plt.tight_layout()
plt.show()